# Description
We aim to compare the process of creating a PD model using a logistic regression and a common ML method (LGBM - Ligh Gradient Boosting Machine)
To do so, this notebook will prepare the data, do the relevant data transformations and create train / test / holdout datasets.

**Modification**: This notebook uses a simplified WoE binning strategy. Numerical variables are binned into deciles, and categorical variables use their unique values as bins.

# Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
import joblib
import os

In [2]:
pd.set_option('display.max_rows', 1000)

# Data

In [5]:
project_folder = "03_deciles_pipeline"

if not os.path.exists(f"../../data/outputs/{project_folder}"):
    os.makedirs(f"../../data/outputs/{project_folder}")

input_path = f"../../data/inputs/02_base_pipeline/"
output_path = f"../../data/outputs/{project_folder}/"

In [6]:
data = pd.read_parquet(f'{input_path}Public_Dataset_Loans.parquet')

# Analysis

## Target Identification & Data Cleaning

In [8]:
def withDefaultFlag(df: pd.DataFrame):
    result = df.copy()
    result['default_flag'] = np.where(result['loan_status'].isin(['Charged Off', 'Default']), 1, 
                                      np.where(result['loan_status'].isin(['Fully Paid']), 0, -1))
    result = result[result['default_flag'].isin([1,0])]
    # Drop loan_status here after using it
    result = result.drop('loan_status', axis=1)
    return result

# Remove 'loan_status' from this list
remove_cols = [
    'url', 'emp_title', 'title', 'zip_code', 'issue_d', 'last_credit_pull_d',
    'last_pymnt_d', 'earliest_cr_line', 'id', 'Unnamed: 0'  # removed 'loan_status'
]

cols_obj_to_float = ['revol_util', 'int_rate']

data_clean = data \
    .pipe(removeNullFeatures, 0.4) \
    .pipe(removeCols, remove_cols) \
    .pipe(withObjtoFloat, cols_obj_to_float) \
    .pipe(withDefaultFlag)

## Train Test Split
With the final dataset filtered and the starter features we will do the train/test split

In [9]:
train_df, test_df = train_test_split(data_clean, train_size=0.60, random_state=42, stratify=data_clean['default_flag'])

## Feature segmentation and Woe (Simplified)

This section introduces a simplified binner. 
- **Numerical variables**: Binned into deciles using `pd.qcut`.
- **Categorical variables**: Each category is treated as a separate bin using `groupby`.

In [10]:
class SimpleBinner:
    def __init__(self, iv_threshold: float = 0.02):
        self.iv_threshold = iv_threshold
        self.woe_maps = {}
        self.iv_scores = {}
        self.selected_cols = []

    def _calculate_woe_iv(self, df, col, target):
        # Group data
        grouped = df.groupby(col, observed=False)[target].agg(['count', 'sum']).reset_index()
        grouped.rename(columns={'sum': 'defaults'}, inplace=True)
        grouped['non_defaults'] = grouped['count'] - grouped['defaults']

        # Calculate totals
        total_defaults = grouped['defaults'].sum()
        total_non_defaults = grouped['non_defaults'].sum()

        # Avoid division by zero
        if total_defaults == 0 or total_non_defaults == 0:
            return None, 0

        # Calculate percentage of defaults and non-defaults
        grouped['perc_defaults'] = grouped['defaults'] / total_defaults
        grouped['perc_non_defaults'] = grouped['non_defaults'] / total_non_defaults

        # Clip to avoid log(0)
        grouped['perc_defaults'] = np.clip(grouped['perc_defaults'], 0.00001, 0.99999)
        grouped['perc_non_defaults'] = np.clip(grouped['perc_non_defaults'], 0.00001, 0.99999)

        # Calculate WoE and IV
        grouped['woe'] = np.log(grouped['perc_defaults'] / grouped['perc_non_defaults'])
        iv = ((grouped['perc_defaults'] - grouped['perc_non_defaults']) * grouped['woe']).sum()
        
        woe_map = grouped.set_index(col)['woe'].to_dict()
        return woe_map, iv

    def fit(self, df, cols_to_bin, target_col):
        self.target_col = target_col
        
        for col in cols_to_bin:
            print(f"Processing: {col}")
            temp_df = df[[col, target_col]].copy()
            
            # Handle numerical vs categorical
            if pd.api.types.is_numeric_dtype(temp_df[col]):
                # Fill NaNs before binning
                if temp_df[col].isnull().any():
                    temp_df[col].fillna(temp_df[col].median(), inplace=True)
                
                # Create deciles, handle non-unique edges
                try:
                    temp_df[col + '_bin'] = pd.qcut(temp_df[col], 10, labels=False, duplicates='drop')
                    bin_col = col + '_bin'
                except ValueError:
                    print(f"  Could not create 10 bins for {col}, skipping.")
                    continue
            else:
                # Fill NaNs for categorical
                if temp_df[col].isnull().any():
                    temp_df[col].fillna('Missing', inplace=True)
                bin_col = col

            woe_map, iv = self._calculate_woe_iv(temp_df, bin_col, target_col)
            
            if woe_map is None:
                print(f"  WoE calculation failed for {col}, skipping.")
                continue

            print(f"  IV: {iv:.4f}")
            if iv > self.iv_threshold:
                self.woe_maps[col] = woe_map
                self.iv_scores[col] = iv
                self.selected_cols.append(col)
                print(f"  >> Selected {col}")
            else:
                print(f"  >> Dropped {col} (IV below threshold)")
        
        self._handle_correlations()
        return self

    def _handle_correlations(self, corr_threshold=0.7):
        print("\nChecking for highly correlated features...")
        woe_df = self.transform(train_df[self.selected_cols])
        corr_matrix = woe_df.corr().abs()
        
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        to_drop = set()
        for column in upper.columns:
            high_corr_features = upper.index[upper[column] > corr_threshold].tolist()
            if high_corr_features:
                for feature in high_corr_features:
                    # Compare IV and keep the one with higher IV
                    if self.iv_scores.get(column, 0) < self.iv_scores.get(feature, 0):
                        to_drop.add(column.replace('_woe', ''))
                    else:
                        to_drop.add(feature.replace('_woe', ''))
        
        if to_drop:
            print(f"Dropping due to high correlation: {list(to_drop)}")
            self.selected_cols = [col for col in self.selected_cols if col not in to_drop]
            self.woe_maps = {k: v for k, v in self.woe_maps.items() if k not in to_drop}
            self.iv_scores = {k: v for k, v in self.iv_scores.items() if k not in to_drop}
        else:
            print("No features dropped due to high correlation.")

    def transform(self, df):
        df_transformed = df.copy()
        for col in self.selected_cols:
            woe_col_name = col + '_woe'
            
            if pd.api.types.is_numeric_dtype(df_transformed[col]):
                # Apply binning first
                bin_col_name = col + '_bin'
                df_transformed[col].fillna(df_transformed[col].median(), inplace=True)
                df_transformed[bin_col_name] = pd.qcut(df_transformed[col], 10, labels=False, duplicates='drop')
                # Then map WoE
                df_transformed[woe_col_name] = df_transformed[bin_col_name].map(self.woe_maps[col])
            else:
                df_transformed[col].fillna('Missing', inplace=True)
                df_transformed[woe_col_name] = df_transformed[col].map(self.woe_maps[col])
            
            # Fill any potential NaNs in WoE column with a neutral value (e.g., 0)
            if df_transformed[woe_col_name].isnull().any():
                df_transformed[woe_col_name].fillna(0, inplace=True)
                
        return df_transformed[[col + '_woe' for col in self.selected_cols]]

#### Run on all variables

In [11]:
cols_to_bin = [
    'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate',
    'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
    'annual_inc', 'verification_status', 'pymnt_plan', 'purpose', 'addr_state',
    'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'initial_list_status'
]

simple_binner = SimpleBinner(iv_threshold=0.02)
simple_binner.fit(train_df, cols_to_bin, 'default_flag')

Processing: loan_amnt
  IV: 0.0322
  >> Selected loan_amnt
Processing: funded_amnt
  IV: 0.0323
  >> Selected funded_amnt
Processing: funded_amnt_inv
  IV: 0.0320
  >> Selected funded_amnt_inv
Processing: term
  IV: 0.1717
  >> Selected term
Processing: int_rate
  IV: 0.4294
  >> Selected int_rate
Processing: installment
  IV: 0.0275
  >> Selected installment
Processing: grade
  IV: 0.4575
  >> Selected grade
Processing: sub_grade
  IV: 0.4906
  >> Selected sub_grade
Processing: emp_length


C:\Users\MY427UW\AppData\Local\Temp\ipykernel_21684\152234913.py:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp_df[col].fillna('Missing', inplace=True)


  IV: 0.0122
  >> Dropped emp_length (IV below threshold)
Processing: home_ownership
  IV: 0.0308
  >> Selected home_ownership
Processing: annual_inc
  IV: 0.0321
  >> Selected annual_inc
Processing: verification_status
  IV: 0.0628
  >> Selected verification_status
Processing: pymnt_plan
  IV: 0.0000
  >> Dropped pymnt_plan (IV below threshold)
Processing: purpose
  IV: 0.0188
  >> Dropped purpose (IV below threshold)
Processing: addr_state
  IV: 0.0168
  >> Dropped addr_state (IV below threshold)
Processing: dti
  IV: 0.0923
  >> Selected dti
Processing: delinq_2yrs
  IV: 0.0023
  >> Dropped delinq_2yrs (IV below threshold)
Processing: fico_range_low
  IV: 0.1367
  >> Selected fico_range_low
Processing: fico_range_high


C:\Users\MY427UW\AppData\Local\Temp\ipykernel_21684\152234913.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp_df[col].fillna(temp_df[col].median(), inplace=True)


  IV: 0.1367
  >> Selected fico_range_high
Processing: inq_last_6mths
  IV: 0.0167
  >> Dropped inq_last_6mths (IV below threshold)
Processing: open_acc
  IV: 0.0072
  >> Dropped open_acc (IV below threshold)
Processing: pub_rec
  IV: 0.0014
  >> Dropped pub_rec (IV below threshold)
Processing: revol_bal


C:\Users\MY427UW\AppData\Local\Temp\ipykernel_21684\152234913.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp_df[col].fillna(temp_df[col].median(), inplace=True)


  IV: 0.0029
  >> Dropped revol_bal (IV below threshold)
Processing: revol_util
  IV: 0.0314
  >> Selected revol_util
Processing: total_acc
  IV: 0.0013
  >> Dropped total_acc (IV below threshold)
Processing: initial_list_status
  IV: 0.0015
  >> Dropped initial_list_status (IV below threshold)

Checking for highly correlated features...


C:\Users\MY427UW\AppData\Local\Temp\ipykernel_21684\152234913.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp_df[col].fillna(temp_df[col].median(), inplace=True)
C:\Users\MY427UW\AppData\Local\Temp\ipykernel_21684\152234913.py:115: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

Dropping due to high correlation: ['fico_range_low', 'grade', 'loan_amnt', 'funded_amnt', 'int_rate', 'funded_amnt_inv']


#### Apply WoE Transformation and Save Data

In [12]:
selected_features = simple_binner.selected_cols
woe_cols = [f"{col}_woe" for col in selected_features]

# Transform train and test sets
train_df_woe = simple_binner.transform(train_df)
test_df_woe = simple_binner.transform(test_df)

# Add the target variable back for modeling
train_df_woe['default_flag'] = train_df['default_flag'].values
test_df_woe['default_flag'] = test_df['default_flag'].values

print(f"Final selected features ({len(woe_cols)}):\n{woe_cols}")

C:\Users\MY427UW\AppData\Local\Temp\ipykernel_21684\152234913.py:120: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_transformed[col].fillna('Missing', inplace=True)
C:\Users\MY427UW\AppData\Local\Temp\ipykernel_21684\152234913.py:115: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

Final selected features (9):
['term_woe', 'installment_woe', 'sub_grade_woe', 'home_ownership_woe', 'annual_inc_woe', 'verification_status_woe', 'dti_woe', 'fico_range_high_woe', 'revol_util_woe']


C:\Users\MY427UW\AppData\Local\Temp\ipykernel_21684\152234913.py:115: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_transformed[col].fillna(df_transformed[col].median(), inplace=True)
C:\Users\MY427UW\AppData\Local\Temp\ipykernel_21684\152234913.py:115: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

In [13]:
# Save the datasets and the binner object
train_df_woe.to_parquet(f'{output_path}train_df_woe.parquet')
test_df_woe.to_parquet(f'{output_path}test_df_woe.parquet')
joblib.dump(simple_binner, f'{output_path}simple_binner.gz')

['../../data/outputs/03_deciles_pipeline/simple_binner.gz']